# 06 · Schema and partition evolution

**Time:** ~25 min · **You will learn:** change a table's columns and partitioning without rewriting data.

Iceberg tracks columns by **id**, not by name or position. That is why renames, reorders and drops are safe metadata-only operations.

In [ ]:
from helpers import get_spark, reset_orders

spark = get_spark("06-evolution")
reset_orders(spark, "orders", n=6, commits=1)
spark.table("demo.course.orders").printSchema()

## 1. Add, rename, widen and drop columns

In [ ]:
spark.sql("ALTER TABLE demo.course.orders ADD COLUMN country STRING")
spark.sql("ALTER TABLE demo.course.orders RENAME COLUMN product TO item")
spark.sql("ALTER TABLE demo.course.orders ALTER COLUMN quantity TYPE BIGINT")   # int -> long is a safe widening
spark.table("demo.course.orders").printSchema()

In [ ]:
spark.table("demo.course.orders").select("order_id", "item", "quantity", "country").show()

The old rows were **not rewritten**. The new column simply reads as `NULL` for them, and `item` still reads the
old `product` data because the column id did not change.

In [ ]:
spark.sql("UPDATE demo.course.orders SET country = 'DE' WHERE order_id IN (1001, 1002)")
spark.sql("ALTER TABLE demo.course.orders DROP COLUMN unit_price")
spark.table("demo.course.orders").show()

## 2. Partitioning without the pain
Partitioning groups rows into folders so queries can skip data. Iceberg uses **hidden partitioning**: you partition by a
*transform* of a column (`days(order_ts)`), and queries just filter on `order_ts`. No extra "date" column to maintain.

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

spark.sql("DROP TABLE IF EXISTS demo.course.events PURGE")
spark.sql("""
    CREATE TABLE demo.course.events (
        event_id BIGINT, user_id BIGINT, event_type STRING, event_ts TIMESTAMP
    ) USING iceberg
    PARTITIONED BY (days(event_ts))
""")

start = datetime(2024, 3, 1)
rows = [(i, i % 50, ["view", "click", "buy"][i % 3], start + timedelta(hours=i * 3)) for i in range(120)]
spark.createDataFrame(rows, "event_id long, user_id long, event_type string, event_ts timestamp") \
     .sortWithinPartitions("event_ts").writeTo("demo.course.events").append()

spark.sql("SELECT partition, record_count, file_count FROM demo.course.events.partitions ORDER BY partition").show(5, truncate=False)

A filter on the raw timestamp prunes partitions automatically:

In [ ]:
q = spark.sql("SELECT COUNT(*) AS n FROM demo.course.events WHERE event_ts >= '2024-03-05' AND event_ts < '2024-03-06'")
q.show()
q.explain()

## 3. Partition evolution
Data grows and access patterns change. You can change the partitioning **for new data**; old files keep their old layout
and Iceberg plans queries across both.

In [ ]:
spark.sql("ALTER TABLE demo.course.events ADD PARTITION FIELD bucket(4, user_id)")

more = [(1000 + i, i % 50, "view", datetime(2024, 3, 10) + timedelta(hours=i)) for i in range(40)]
spark.createDataFrame(more, "event_id long, user_id long, event_type string, event_ts timestamp") \
     .sortWithinPartitions("event_ts", "user_id").writeTo("demo.course.events").append()

spark.sql("SELECT spec_id, COUNT(*) AS files FROM demo.course.events.files GROUP BY spec_id ORDER BY spec_id").show()

Two partition specs now coexist (`spec_id` 0 and 1), and one query reads both transparently:

In [ ]:
spark.sql("SELECT COUNT(*) AS total_events FROM demo.course.events").show()

### Your turn

On `demo.course.orders`, add a column `status STRING`, set it to `'shipped'` for every order placed before
`2024-01-02`, and assert that older rows show it while the others are `NULL`.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
spark.sql("ALTER TABLE demo.course.orders ADD COLUMN status STRING")
spark.sql("UPDATE demo.course.orders SET status = 'shipped' WHERE order_ts < TIMESTAMP '2024-01-02 00:00:00'")
df = spark.table("demo.course.orders")
shipped = df.filter("status = 'shipped'").count()
pending = df.filter("status IS NULL").count()
print(shipped, pending)
assert shipped >= 1 and pending >= 1 and shipped + pending == df.count()
```

</details>

### Recap
* `ADD / RENAME / DROP / ALTER COLUMN TYPE` are metadata operations. No data rewrite, and ids keep old files valid.
* Hidden partitioning (`days(ts)`, `bucket(n, col)`, ...) lets queries prune without knowing the layout.
* Partition specs can evolve; old and new data coexist.

Next: **07 · PyIceberg: reading tables**.